# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Moezulhaq24/FlyRank-Internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install -q datasets huggingface_hub duckdb
!pip install duckdb pyarrow

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

In [3]:
from huggingface_hub import login

login(HF_TOKEN)

In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    name='fact_content_daily_performance',
    token=HF_TOKEN
)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

In [5]:
dataset.keys()

dict_keys(['train'])

In [6]:
import duckdb

The cell is taking so much time to run.

In [ ]:
import pandas as pd

hf_dataset_df = dataset['train'].to_pandas()

query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM hf_dataset_df  -- Querying the pandas DataFrame
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
LIMIT 10000
"""

df = duckdb.query(query).to_df()

df.head()

In [ ]:
df.head()

In [7]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})


In [9]:
print(dataset["train"])

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 78835655
})


In [10]:
print(type(dataset["train"]))

<class 'datasets.arrow_dataset.Dataset'>


In [6]:
print(dataset["train"].cache_files)

[{'filename': '/root/.cache/huggingface/datasets/FlyRank___internship-warehouse/fact_content_daily_performance/0.0.0/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/internship-warehouse-train-00000-of-00039.arrow'}, {'filename': '/root/.cache/huggingface/datasets/FlyRank___internship-warehouse/fact_content_daily_performance/0.0.0/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/internship-warehouse-train-00001-of-00039.arrow'}, {'filename': '/root/.cache/huggingface/datasets/FlyRank___internship-warehouse/fact_content_daily_performance/0.0.0/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/internship-warehouse-train-00002-of-00039.arrow'}, {'filename': '/root/.cache/huggingface/datasets/FlyRank___internship-warehouse/fact_content_daily_performance/0.0.0/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/internship-warehouse-train-00003-of-00039.arrow'}, {'filename': '/root/.cache/huggingface/datasets/FlyRank___internship-warehouse/fact_content_daily_performance/0.0.0/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/internship

## 1. Unit of analysis + time window

**One row** represents the daily performance of one content page for one client on one reporting date.

Grain:
1 row = 1 content page + 1 client + 1 day

**Primary Table:**

fact_content_daily_performance

This table contains the daily Search Console and Google Analytics performance signals required for page-level refresh opportunity analysis.

The table provides historical page performance that can later be transformed into features for machine learning.

**Time Window:**

For this assignment I will use a mid-panel month (March 2026).

Using a middle month avoids using the final month of the dataset as training data and follows the internship guidance to keep the last month as a future evaluation period.

**Prediction / Ranking Target:**

The business goal is to help SEO analysts prioritize which pages should receive limited refresh or review slots.

The final system will rank pages according to their refresh opportunity.

Since "worth refreshing" is not directly measurable, a future decline or refresh-worthiness proxy will be used during modeling.

**What is deliberately excluded?**

The following information is deliberately excluded:

• Future performance information (prevents data leakage)
• Client identity
• Page URLs
• Any future observations after the prediction date

The model should only use information available at the time the refresh decision is made.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

### Feature Fields

These fields are available before making the decision and describe historical page performance.

• gsc_impressions

• gsc_clicks

• gsc_avg_position

• ga4_engaged_sessions

• ga4_total_engagement_sec

---

### Label

Future decline (proxy)

The actual business goal is refresh opportunity, but this is not directly measurable. Therefore a future decline proxy will later be used for supervised learning.

---

### Context Fields

These identify each observation but are not model inputs.

• report_date

• client_hash_id

• content_hash_id

---

### Excluded Fields

• client_has_gsc

• client_has_ga4

• gsc_data_available

• ga4_data_available


Reason:
These fields describe data availability rather than page quality or performance. They are useful for filtering and validation but not as predictive features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 (Grain Verification)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Filter March 2026

march = df[df["report_date"].astype(str).str.startswith("2026-03")]

print("Total Rows:", len(march))

unique_rows = march[
    ["report_date", "client_hash_id", "content_hash_id"]
].drop_duplicates()

print("Unique Grain:", len(unique_rows))

if len(unique_rows) == len(march):
    print("✅ Grain Verified")
else:
    print("❌ Duplicate Rows Found")

### Query 2

In [ ]:
print("Row Count:", len(march))

print("Start Date:", march["report_date"].min())

print("End Date:", march["report_date"].max())

### Query 3

In [ ]:
available = march[march["gsc_data_available"] == True]

print("Available Rows:", len(available))

print(
    "Percentage:",
    round(len(available) / len(march) * 100, 2),
    "%"
)

## 4. Data limits

This data has several limitations.

• The warehouse records historical performance but cannot directly measure content quality.

• A page recommended for refresh is not guaranteed to recover traffic after being updated.

• Some observations may not contain Google Search Console or Google Analytics data.

• The selected month represents only one period and may not capture seasonal behaviour.

• This dataset supports decision-making but cannot prove causal relationships.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.